# Performance Assessment of Best Models (CatBoost (CB) Feats+Text, CatBoost Text Only, CatBoost Tabular SMOTE, LR)

This notebook is for assessing the performance of the best models, including threshold metrics, confusion matrices, and ROC/PR curves. It includes code to load the models, generate the plots, and save them to the appropriate directories.

## Import Requisite Libraries

In [ ]:
import pandas as pd
from core.constants import drop_vars

## Paths

In [ ]:
data_path = "../data/processed"

In [ ]:
from eda_toolkit import ensure_directory
import os # import operating system for dir


base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

## Load the best models from the dramatic_model experiment.

In [ ]:
import json
from pathlib import Path

pred_dir = Path("../models/predictions")

# Load pre-saved predictions — run modeling/save_predictions.py first
# if this directory does not exist.
y_test = pd.read_parquet(pred_dir / "y_test.parquet").squeeze()

y_prob_cat_feats_and_text = pd.read_parquet(
    pred_dir / "y_prob_cat_feats_and_text.parquet"
).squeeze()

y_prob_cat_text_only = pd.read_parquet(
    pred_dir / "y_prob_cat_text_only.parquet"
).squeeze()

y_prob_cat = pd.read_parquet(
    pred_dir / "y_prob_cat.parquet"
).squeeze()

y_prob_lr = pd.read_parquet(
    pred_dir / "y_prob_lr.parquet"
).squeeze()

with open(pred_dir / "model_thresholds.json") as f:
    model_thresholds = json.load(f)

In [ ]:
from model_tuner import loadObjects

# # Best overall -- CB Feats+Text, AUC 0.814 (experiment 724492581979628771)
model_cat_feats_and_text = loadObjects(
    "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/567790926067075047/6f88542d9d4c4809bfd097ccc22cc3cd/artifacts/cat_feats_and_text_dramatic/model.pkl"
)

# Best text only -- CB Text Only, AUC 0.709 (experiment 724492581979628771)
model_cat_text_only = loadObjects(
    "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/567790926067075047/b3bff7e23ed446faaa871e5c99cf1429/artifacts/cat_text_only_dramatic/model.pkl"
)

# # Best tabular CatBoost -- cat smote, AUC 0.807 (experiment 332182865953453352)
# model_cat = loadObjects(
#     "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/332182865953453352/9b1f437304b74cd8998692ec10179634/artifacts/cat_dramatic/model.pkl"
# )

# # Best LR -- lr orig, AUC 0.791 (experiment 332182865953453352)
# model_lr = loadObjects(
#     "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/332182865953453352/1d3dafbcf4734f269c5591c0116ab2a7/artifacts/lr_dramatic/model.pkl"
# )

## Threshold Values Extraction

In [ ]:
model_thresholds

In [ ]:
# model_thresholds = {
#     "CatBoost Feats + Text":    next(iter(model_cat_feats_and_text.threshold.values())),
#     "CatBoost Text Only":       next(iter(model_cat_text_only.threshold.values())),
#     "CatBoost Tabular (SMOTE)": next(iter(model_cat.threshold.values())),
#     "Logistic Regression":      next(iter(model_lr.threshold.values())),
# }a

## Load test data for final evaluation of best models

In [ ]:
X = pd.read_parquet(os.path.join(data_path, "X.parquet"))
y = pd.read_parquet(os.path.join(data_path, "y_dramatic.parquet"))

In [ ]:
# X_full = pd.read_parquet("../data/processed/X.parquet")
# y = pd.read_parquet("../data/processed/y_dramatic.parquet").squeeze()

# # cat_feats_and_text
# X_feats_and_text = X_full.drop(columns=[c for c in drop_vars if c in X_full.columns], errors="ignore")
# X_feats_and_text["summary_clean"] = X_feats_and_text["summary_clean"].fillna("").astype(str)

# # cat_text_only
# X_text_only = X_full[["summary_clean"]].copy()
# X_text_only["summary_clean"] = X_text_only["summary_clean"].fillna("").astype(str)

# # tabular (cat, lr)
# X_tabular = X_full.drop(columns=["summary_clean"], errors="ignore")

# # Now pull test splits from each model
# X_test_feats_and_text, y_test = model_cat_feats_and_text.get_test_data(X_feats_and_text, y)
# X_test_text_only, _           = model_cat_text_only.get_test_data(X_text_only, y)
# X_test_cat, _                 = model_cat.get_test_data(X_tabular, y)
# X_test_lr, _                  = model_lr.get_test_data(X_tabular, y)

# y_prob_cat_feats_and_text = model_cat_feats_and_text.predict_proba(X_test_feats_and_text)[:, 1]
# y_prob_cat_text_only      = model_cat_text_only.predict_proba(X_test_text_only)[:, 1]
# y_prob_cat                = model_cat.predict_proba(X_test_cat)[:, 1]
# y_prob_lr                 = model_lr.predict_proba(X_test_lr)[:, 1]

In [ ]:
X_train, y_train = model_cat_feats_and_text.get_train_data(X,y)
X_test, y_test = model_cat_feats_and_text.get_test_data(X,y)

In [ ]:
X_test_cat_feats_and_text = pd.read_parquet(pred_dir / "X_test_cat_feats_and_text.parquet")
# etc.

In [ ]:
with open(pred_dir / "model_feature_names.json") as f:
    feature_names = json.load(f)

# e.g. feature_names["cat_feats_and_text"]

In [ ]:
feature_names

In [ ]:
for model, feats in feature_names.items():
    print(f"{model}: {len(feats)} features")
# len(feature_names["cat_feats_and_text"])

## ROC AUC and Precision-Recall AUC for Regular Features Plus Combined Text

In [ ]:
from model_metrics import combine_plots, show_roc_curve, show_pr_curve
combine_plots(
    plot_calls=[
        (
            show_roc_curve,
            {
                "y_prob": [
                    y_prob_cat_feats_and_text,
                    y_prob_cat_text_only,
                    y_prob_cat,
                    y_prob_lr,
                ],
                "y": y_test,
                "model_title": [
                    "CatBoost Feats + Text",
                    "CatBoost Text Only",
                    "CatBoost Tabular (SMOTE)",
                    "Logistic Regression",
                ],
                "overlay": True,
                "legend_loc": "bottom",
                "decimal_places": 2,
                "title": "AUROC Curves: CatBoost vs. Logistic Regression",
            },
        ),
        (
            show_pr_curve,
            {
                "y_prob": [
                    y_prob_cat_feats_and_text,
                    y_prob_cat_text_only,
                    y_prob_cat,
                    y_prob_lr,
                ],
                "y": y_test,
                "model_title": [
                    "CatBoost Feats + Text",
                    "CatBoost Text Only",
                    "CatBoost Tabular (SMOTE)",
                    "Logistic Regression",
                ],
                "overlay": True,
                "legend_loc": "bottom",
                "decimal_places": 2,
                "title": "Precision-Recall: CatBoost vs. Logistic Regression",
            },
        ),
    ],
    n_cols=2,
    suptitle="All Models: ROC & PR Curves (Dramatic Outcome)",
    suptitle_y=1.18,
    hspace=0.5,
    image_path_png=image_path_png,
    image_path_svg=image_path_svg,
    image_filename="all_models_roc_pr_curves_dramatic",
    label_fontsize=14,
    tick_fontsize=12,
    wspace=0.1,
    figsize=(12, 6)
)

## Confusion Matrices

In [ ]:
from model_metrics import show_confusion_matrix

combine_plots(
    plot_calls=[
        (
            show_confusion_matrix,
            {
                "y_prob": y_prob_cat_feats_and_text.values,
                "y": y_test.values,
                "model_title": "CatBoost Feats + Text",
                "inner_fontsize": 14,
                # "text_wrap": 40,
                "model_threshold": model_thresholds["CatBoost Feats + Text"],
                "decimal_places": 2,
                "inner_fontsize":14,
                "title": "CatBoost Feats + Text\n(Threshold= {:.2f})".format(model_thresholds["CatBoost Feats + Text"]),
            },
        ),
        (
            show_confusion_matrix,
            {
                "y_prob": y_prob_cat_text_only.values,
                "y": y_test.values,
                "model_title": "CatBoost Text Only",
                "inner_fontsize": 14,
                # "text_wrap": 40,
                "model_threshold": model_thresholds["CatBoost Text Only"],
                "decimal_places": 2,
                "inner_fontsize":14,
                "title": "CatBoost Text Only\n(Threshold= {:.2f})".format(model_thresholds["CatBoost Text Only"]),
            },
        ),
        (
            show_confusion_matrix,
            {
                "y_prob": y_prob_cat.values,
                "y": y_test.values,
                "model_title": "CatBoost Tabular (SMOTE)",
                "inner_fontsize": 14,
                # "text_wrap": 40,
                "model_threshold": model_thresholds["CatBoost Tabular (SMOTE)"],
                "decimal_places": 2,
                "inner_fontsize":14,
                "title": "CatBoost Tabular (SMOTE)\n(Threshold= {:.2f})".format(model_thresholds["CatBoost Tabular (SMOTE)"]),
            },
        ),
        (
            show_confusion_matrix,
            {
                "y_prob": y_prob_lr.values,
                "y": y_test.values,
                "model_title": "Logistic Regression",
                "inner_fontsize": 14,
                # "text_wrap": 40,
                "model_threshold": model_thresholds["Logistic Regression"],
                "decimal_places": 2,
                "inner_fontsize":14,
                "title": "Logistic Regression\n(Threshold= {:.2f})".format(model_thresholds["Logistic Regression"]),
            },
        ),
    ],
    image_path_png=image_path_png,
    image_path_svg=image_path_svg,
    label_fontsize=14,
    tick_fontsize=12,
    suptitle="All Models: Confusion Matrices (Dramatic Outcome)",
    suptitle_y=1.02,
    image_filename="all_models_confusion_matrices_dramatic",
)

## Calibration Curves of Best Models (CB Feats+Text, CB Text Only, CatBoost Tabular SMOTE, LR)

In [ ]:
from model_metrics import show_calibration_curve

show_calibration_curve(
    y_prob=[
        y_prob_cat_feats_and_text,
        y_prob_cat_text_only,
        y_prob_cat,
        y_prob_lr,
    ],
    y=y_test,
    model_title=[
        "CatBoost Feats + Text",
        "CatBoost Text Only",
        "CatBoost Tabular (SMOTE)",
        "Logistic Regression",
    ],
    title="Calibration Curves of Best Models (Dramatic Outcome)",
    show_brier_score=True,
    overlay=True,
    legend_loc="bottom",
    brier_decimals=3,
    figsize=(12, 6),
    tick_fontsize=12,
    label_fontsize=14,
    image_path_png=image_path_png,
    image_path_svg=image_path_svg,
    image_filename="all_models_calibration_curves_dramatic",
)

## Threshold Metrics of Best Models (CB Feats+Text, CB Text Only, CatBoost Tabular SMOTE, LR)

In [ ]:
from model_metrics import plot_threshold_metrics

combine_plots(
    plot_calls=[
        (
            plot_threshold_metrics,
            {
                "y_prob": y_prob_cat_feats_and_text.values,
                "y_test": y_test,
                "title": "CatBoost Feats + Text - Threshold Metrics",
                "baseline_thresh": False,
                "model_threshold": model_thresholds["CatBoost Feats + Text"],
                "threshold_kwgs": {
                    "color": "black",
                    "linestyle": "--",
                    "linewidth": 2,
                },
                "curve_kwgs": {"linestyle": "-", "linewidth": 2},
                "decimal_places": 2,
            },
        ),
        (
            plot_threshold_metrics,
            {
                "y_prob": y_prob_cat_text_only.values,
                "y_test": y_test,
                "title": "CatBoost Text Only - Threshold Metrics",
                "baseline_thresh": False,
                "model_threshold": model_thresholds["CatBoost Text Only"],
                "threshold_kwgs": {
                    "color": "black",
                    "linestyle": "--",
                    "linewidth": 2,
                },
                "curve_kwgs": {"linestyle": "-", "linewidth": 2},
                "decimal_places": 2,
            },
        ),
        (
            plot_threshold_metrics,
            {
                "y_prob": y_prob_cat.values,
                "y_test": y_test,
                "title": "CatBoost Tabular (SMOTE) - Threshold Metrics",
                "baseline_thresh": False,
                "model_threshold": model_thresholds["CatBoost Tabular (SMOTE)"],
                "threshold_kwgs": {
                    "color": "black",
                    "linestyle": "--",
                    "linewidth": 2,
                },
                "curve_kwgs": {"linestyle": "-", "linewidth": 2},
                "decimal_places": 2,
            },
        ),
        (
            plot_threshold_metrics,
            {
                "y_prob": y_prob_lr.values,
                "y_test": y_test,
                "title": "Logistic Regression - Threshold Metrics",
                "baseline_thresh": False,
                "model_threshold": model_thresholds["Logistic Regression"],
                "threshold_kwgs": {
                    "color": "black",
                    "linestyle": "--",
                    "linewidth": 2,
                },
                "curve_kwgs": {"linestyle": "-", "linewidth": 2},
                "decimal_places": 2,
            },
        ),
    ],
    image_path_png=image_path_png,
    image_path_svg=image_path_svg,
    image_filename="all_models_threshold_metrics_dramatic",
    figsize=(14, 10),
    tick_fontsize=12,
    label_fontsize=14,
)

## SHAP Values

In [ ]:
from core.functions import create_shap_plots
from pathlib import Path
import matplotlib.pyplot as plt

shap_importance_expanded, shap_importance, shap_figs = create_shap_plots(
    model=model_cat_feats_and_text,
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    output_dir=Path(image_path_svg),
    max_display=20,
    sample_size=500,
    # feature_rename=feature_rename,
    side_by_side=True,
)

# # Rename features in the returned DataFrames
# shap_importance["feature"] = shap_importance["feature"].apply(clean_feature_name)
# shap_importance_expanded["feature"] = shap_importance_expanded["feature"].apply(
#     clean_feature_name
# )

for name, fig in shap_figs.items():
    fig.set_size_inches(10, 15)   # adjust per plot if needed
    fig.tight_layout()
    fig.show()

plt.show()